In [1]:
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import pandas as pd
print("Module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("Path append success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund_gable")
env
print("Bootstrap environment success")

from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")
print("Utilities import success")

import matplotlib.pyplot as plt
import requests

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
processed_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data")

c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module import success
Path append success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 9, 13, 52, 8, 283804),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESO

In [14]:
#Helper function that opens a DO dataset and drops any rows with missing DO values
def open_DO_dataset(file_path, DO_column_name):
    ds=xr.open_dataset(file_path)
    df=ds.to_dataframe()
    df=df.reset_index()
    df=df.dropna(subset=[DO_column_name])
    return df

In [61]:
# 3. Define the labeling function

def classify_DO(df, DO_column, battery_column=None):
    classified_df = df.copy()

    classified_df.loc[classified_df[DO_column] >= 3.9, 'DO_status'] = 'Low Oxygen'
    classified_df.loc[classified_df[DO_column] > 5.0, 'DO_status'] = 'Oxygenated' 
    classified_df.loc[classified_df[DO_column] < 3.9, 'DO_status'] = 'Mildly Hypoxic'
    classified_df.loc[classified_df[DO_column] < 2.41, 'DO_status'] = 'Hypoxia'
    classified_df.loc[classified_df[DO_column] < 0.2, 'DO_status'] = 'Anoxia'
    
    # 2. FILTER BATTERY (Only if a battery column name was provided)
    if battery_column is not None:
        classified_df = classified_df[classified_df[battery_column] >= 50]
        
    classified_df = classified_df.reset_index(drop=True)
    
    return classified_df

# 4. Run the function on your DataFrame
# You can now pass None explicitly, or just leave the third argument completely blank!
#Fishbot_hypoxia = classify_DO(dff, 'dissolved_oxygen')
# 4. Run the function on your DataFrame
#Fishbot_hypoxia = classify_DO(dff, 'dissolved_oxygen', None)

# 5. Print the columns to verify 'depth' is finally there
#print(Fishbot_hypoxia.columns.tolist())
#Fishbot_hypoxia
#Fishbot_hypoxia.to_excel(root / "data" / "Processed_Data" / "Fishbot_Hypoxia.xlsx", index=False)

def get_extent(df, time_column, lon_column, lat_column, depth_column=None):
    # 1. Convert to datetime and strip hours/minutes/seconds (normalizes to midnight)
    daily_time = pd.to_datetime(df[time_column]).dt.normalize()
    
    # 2. Group by the new daily time
    lon_max = df.groupby(daily_time)[lon_column].max()
    lon_min=df.groupby(daily_time)[lon_column].max()
    lat_max = df.groupby(daily_time)[lat_column].max()
    lat_min = df.groupby(daily_time)[lat_column].min()
  
    # 3. Create the dataframe
    if depth_column is not None: 
      depth_max=df.groupby(daily_time)[depth_column].max()
      depth_min=df.groupby(daily_time)[depth_column].max()
      extent_df = pd.DataFrame({
        "lon_max": lon_max,
        "lon_min": lon_min,
        "lat_max": lat_max,
        "lat_min": lat_min,
        "depth_max": depth_max,
        "depth_min": depth_min
    })
    else:
      extent_df = pd.DataFrame({
        "lon_max": lon_max,
        "lon_min": lon_min,
        "lat_max": lat_max,
        "lat_min": lat_min})

    
    # 4. Reset index so 'time_column' becomes a regular column again instead of the index
    return extent_df.reset_index()

def calculate_daily_averages(df, datetime_col, target_col):
    df_working = df.copy()
    
    # 1. Normalize the time to midnight
    df_working['daily_time'] = pd.to_datetime(df_working[datetime_col]).dt.normalize()
    
    # 2. Ensure the target column is numeric
    df_working[target_col] = pd.to_numeric(df_working[target_col], errors='coerce')
    
    # *** THE FIX: Replace any absurdly high fill values with NaN ***
    # You can adjust this threshold (e.g., 50) based on what makes sense for your specific parameter!
    df_working.loc[df_working[target_col] > 50, target_col] = np.nan
    
    # 3. Group by the daily_time (Pandas will automatically ignore the NaNs when calculating the mean)
    daily_avg_df = df_working.groupby('daily_time')[target_col].mean().reset_index()
    
    # 4. Rename 'daily_time' back to the original time column name
    avg_col_name = f'{target_col}_daily_avg'
    daily_avg_df.rename(columns={'daily_time': datetime_col, target_col: avg_col_name}, inplace=True)
    
    return daily_avg_df

def combine_df(df1, df2, join_key1):
    # Make copies so we don't accidentally alter the original dataframes
    df1_working = df1.copy()
    df2_working = df2.copy()
    
    # Force the join key in both dataframes to be a standard Pandas datetime object
    df1_working[join_key1] = pd.to_datetime(df1_working[join_key1])
    df2_working[join_key1] = pd.to_datetime(df2_working[join_key1])
    
    # Now that the data types perfectly match, perform the merge
    combined_df = pd.merge(df1_working, df2_working, on=join_key1)
    
    return combined_df

In [60]:
#1. Open the dataset and check the column headers
ds = xr.open_dataset(root / raw_data / "eMOLT" / "eMOLT_2000_2026.nc")
dfeM = ds.to_dataframe().reset_index() 
print(dfeM.columns.to_list())

#2. calculate daily averages 
dfeMd = calculate_daily_averages(dfeM, 'time', 'DO')

#3. classify the daily averages
dfeMdc = classify_DO(dfeMd, 'DO_daily_avg')

#  Force time to midnight so it matches exactly ***
dfeMdc['time'] = pd.to_datetime(dfeMdc['time']).dt.normalize()

#4. get spatial extent
dfeMe = get_extent(dfeM, 'time', 'longitude', 'latitude') 

#5. Reset the index and ensure the column is named 'time' 
# Use rename just in case the groupby index lost its 'time' name
#dfeMe = dfeMe.reset_index().rename(columns={'index': 'time'})

#5. combine spatial extent with classified daily averages
df_final = combine_df(dfeMdc, dfeMe, 'time')
print(f"Distinct dates after merge: {df_final['time'].nunique()}")

#6. export dataset
export_path = root / "data" / "Processed_Data" / "eMOLT_DO_extent_average_2023_2026.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d', index=False)
#6. export dataset
export_path = root / "data" / "Processed_Data" / "eMOLT_DO_extent_average_2023_2026.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

['row', 'tow_id', 'time', 'latitude', 'longitude', 'temperature', 'DO', 'DO_percentage', 'water_detect_perc', 'sensor_type']
Distinct dates after merge: 767


In [ ]:
dfeM

,time,date,DO_daily_avg
0,2023-07-11 17:32:44,2023-07-11,7.63
1,2023-07-11 17:35:44,2023-07-11,7.30
2,2023-07-11 17:35:58,2023-07-11,7.36
3,2023-07-11 17:36:12,2023-07-11,7.42
4,2023-07-11 17:39:48,2023-07-11,7.29
...,...,...,...
958133,2026-05-24 22:53:49,2026-05-24,10.65
958134,2026-05-24 23:08:49,2026-05-24,10.64
958135,2026-05-24 23:23:49,2026-05-24,10.64
958136,2026-05-24 23:38:49,2026-05-24,10.64


In [92]:
ds=xr.open_dataset(root / "data" / "Raw_Data" / "eMOLT" / "eMOLT_2000_2026.nc")
df=ds.to_dataframe()
print(df["time"].min())
df=df.reset_index()
DO_column_name="DO"
df=drop_na_do(df)
eMOLT=find_low_oxygen(df, threshold=5, mild_hypoxia_threshold=3.9, hypoxia_threshold=2.41, anoxia_threshold=0.2)
#eMOLT.to_excel(root / "data" / "Processed_Data" / "eMOLT_Hypoxia.xlsx", index=False)


2023-07-11 17:32:44


In [18]:
ds=xr.open_dataset(root / raw_data / "Hydrographic_Monitoring" / "errdap_search.nc")
dfh=ds.to_dataframe()
print(dfh.columns.to_list())
dfh
unique_cruise=dfh['cruise_id'].dropna().unique()
print(unique_cruise)
unique_cruise_df=pd.DataFrame([{'Cruise Names':unique_cruise}])
unique_cruise_df.to_excel(root / processed_data / 'hydrographic_monitoring_cruises.xlsx', index=False)




['UTC_DATETIME', 'latitude', 'longitude', 'depth', 'dissolved_oxygen', 'cruise_id', 'purpose_code', 'bottom_depth', 'GEAR_TYPE']
<ArrowStringArray>
['DEL0909', 'DEL0911', 'DEL1001', 'DEL1004', 'DEL1012', 'DEL1105', 'DEL1202',
  'HB1202',  'HB1205',  'HB1206',  'PC1206',  'PC1207',  'PC1301',  'GU1302',
  'HB1303',  'HB1304',  'EX1305',  'GU1305',  'GU1401',  'HB1401',  'HB1403',
  'HB1405',  'PC1405',  'HB1501',  'HB1502',  'HB1503',  'GU1506',  'HB1506',
  'HB1601',  'HB1603',  'HB1604',  'PC1607',  'GU1608',  'PC1609',  'GU1701',
  'HB1701',  'HB1702',  'GU1706',  'S11801',  'HB1802',  'S11802',  'HB1803',
  'GU1804',  'HB1805',  'HB1806',  'HB1901',  'S11901',  'GU1902',  'HB1902',
  'GU1905',  'HB1906',  'HB1907',  'HB2101',  'S12101',  'GU2102',  'HB2103',
  'PC2104',  'PC2106',  'S12201',  'HB2202',  'HB2204',  'PC2205',  'HB2207',
  'HB2301',  'S12301',  'HB2302',  'HB2303',  'HB2304',  'PC2305',  'HB2401',
  'HB2402',  'HB2403',  'HB2406',  'PC2406',  'HB2407',  'HB2501',  'HB2

In [20]:


# 1. Reset index and get unique values, then force them into a standard list
cruise_list = dfh['cruise_id'].dropna().unique().tolist()

# 2. Put them in the DataFrame (Make sure there are NO square brackets around cruise_list!)
df_export = pd.DataFrame({'Cruise Names': cruise_list})

# 3. Export to Excel
df_export.to_excel(root / processed_data / 'hydrographic_monitoring_cruises.xlsx', index=False)